# Optuna Model Inspector

This notebook instantiates and inspects the `TunableEncoderFnlModel` from the Optuna tuning study with the best hyperparameters found during optimization.

## 1. Import Required Libraries

In [ ]:
import os
import sys
import logging

# Suppress TensorFlow logs before import
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Add deepsphere to path
sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

import healpy as hp
import numpy as np

from mlpng import Core
from mlpng.optuna_trainer import TunableEncoderFnlModel, EncoderBlock
from mlpng.utils import setup_logging

# Setup logging for notebook
setup_logging("mlpng.notebook", level=logging.INFO)

## 2. Best Hyperparameters from Optuna Study

These are the best hyperparameters found during the Optuna tuning run:

In [ ]:
# Best hyperparameters from Optuna study
best_params = {
    "use_cosine_decay": True,
    "initial_lr": 0.0006063500256249086,
    "pool_p": 2,
    "transformer_levels": 0,
    "num_heads": 2,
    "transformer_layers": 3,
    "dropout_rate": 0.15000000000000002,
    "weight_decay": 3.777900967315528e-07,
    "dense_units": 32,
    "dense_layers": 1,
    "head_dropout": 0.0,
}

# Display the hyperparameters
print("Best Hyperparameters:")
print("-" * 40)
for key, value in best_params.items():
    print(f"  {key}: {value}")

## 3. Initialize Core Configuration

Load the settings file to get `nside`, `npix`, `npols`, and `shapes` for model input.

In [ ]:
# Initialize Core with settings file
core = Core(["./settings/n256.json", "--shapes", "local", "--nsims", "10000"])

print(f"nside: {core.nside}")
print(f"npix:  {core.npix}")
print(f"npols: {core.npols}")
print(f"shapes: {core.shapes}")
print(f"n_outputs: {len(core.shapes)}")

## 4. Build the Model with Best Hyperparameters

Create the `TunableEncoderFnlModel` using the best hyperparameters from the Optuna study.

In [ ]:
# Build the model with best hyperparameters
batch_size = 32

model = TunableEncoderFnlModel(
    input_shape=(None, core.npix, core.npols),
    n_outputs=len(core.shapes),
    max_batch_size=batch_size,
    pool_p=best_params["pool_p"],
    transformer_levels=best_params["transformer_levels"],
    num_heads=best_params["num_heads"],
    transformer_layers=best_params["transformer_layers"],
    dropout_rate=best_params["dropout_rate"],
    dense_units=best_params["dense_units"],
    dense_layers=best_params["dense_layers"],
    head_dropout=best_params["head_dropout"],
).get_model()

print(f"Model built: {model.name}")

## 5. Model Summary (Standard View)

In [ ]:
# Standard model summary
model.summary()

## 6. Expanded Model Summary (Including Inner Layers)

Use `expand_nested=True` to display the full architecture including the inner layers of each `EncoderBlock`.

In [ ]:
# Expanded model summary showing nested layers
model.summary(expand_nested=True)

## 7. Layer-by-Layer Details

Inspect each layer's configuration and parameter counts.

In [ ]:
def inspect_layer(layer, indent=0):
    """Recursively inspect layers and their sublayers."""
    prefix = "  " * indent
    params = layer.count_params()
    trainable = sum(tf.keras.backend.count_params(w) for w in layer.trainable_weights)

    print(f"{prefix}├─ {layer.name} ({layer.__class__.__name__})")
    print(f"{prefix}│    Parameters: {params:,} (trainable: {trainable:,})")

    if hasattr(layer, "output_shape"):
        try:
            print(f"{prefix}│    Output shape: {layer.output_shape}")
        except:
            pass

    # Check for nested layers
    if hasattr(layer, "layers"):
        for sublayer in layer.layers:
            inspect_layer(sublayer, indent + 1)
    elif hasattr(layer, "body") and hasattr(layer.body, "layers"):
        # For EncoderBlock which wraps HealpyGCNN in .body
        print(f"{prefix}│    Contains HealpyGCNN with sublayers:")
        for sublayer in layer.body.layers:
            inspect_layer(sublayer, indent + 1)


print("Model Layer Hierarchy")
print("=" * 60)
for layer in model.layers:
    inspect_layer(layer)

print("\n" + "=" * 60)
print(f"Total parameters: {model.count_params():,}")

## 8. Architecture Configuration Summary

Display computed architecture details based on the hyperparameters.

In [ ]:
import math

# Compute architecture details from hyperparameters
pool_p = best_params["pool_p"]
nside = core.nside
nside_factor = 2**pool_p
pixel_factor = 4**pool_p

depth = int(math.log(nside, nside_factor))
level_nsides = [nside // (nside_factor**i) for i in range(depth + 1)]
level_npixels = [12 * ns**2 for ns in level_nsides]
channels = [core.npols] + [2 ** (i + 5) for i in range(depth + 1)]

# Transformer placement
transformer_levels = best_params["transformer_levels"]
use_transformer = [False] * depth
if transformer_levels > 0:
    start_idx = max(0, depth - transformer_levels)
    for i in range(start_idx, depth):
        use_transformer[i] = True

print("Architecture Configuration")
print("=" * 60)
print(f"\nEncoder:")
print(f"  Depth (number of encoder blocks): {depth}")
print(f"  Pool power (pool_p): {pool_p} → {nside_factor}x nside reduction per block")
print(f"  Pixel reduction per block: {pixel_factor}x")
print(
    f"  Transformer levels: {transformer_levels} (applied to last {transformer_levels} blocks)"
)

print(f"\nLevel Details:")
print(
    f"  {'Level':<8} {'nside':<8} {'npix':<12} {'channels_in':<12} {'channels_out':<12} {'transformer':<12}"
)
print(f"  {'-'*64}")
for i in range(depth):
    xformer = "Yes" if use_transformer[i] else "No"
    print(
        f"  {i:<8} {level_nsides[i]:<8} {level_npixels[i]:<12} {channels[i]:<12} {channels[i+1]:<12} {xformer:<12}"
    )

print(f"\nDense Head:")
print(f"  Dense units: {best_params['dense_units']}")
print(f"  Dense layers: {best_params['dense_layers']}")
print(f"  Head dropout: {best_params['head_dropout']}")

print(f"\nRegularization:")
print(f"  Encoder dropout: {best_params['dropout_rate']}")
print(f"  Weight decay: {best_params['weight_decay']:.2e}")

print(f"\nOptimizer:")
print(
    f"  Learning rate schedule: {'CosineDecayRestarts' if best_params['use_cosine_decay'] else 'ExponentialDecay'}"
)
print(f"  Initial learning rate: {best_params['initial_lr']:.6f}")

## 9. Model Visualization (Optional)

Generate a visual diagram of the model architecture. Requires `graphviz` and `pydot`.

In [ ]:
# Generate model architecture diagram (requires graphviz and pydot)
try:
    tf.keras.utils.plot_model(
        model,
        to_file="optuna_model_architecture.png",
        show_shapes=True,
        show_layer_names=True,
        expand_nested=True,
        dpi=100,
    )
    print("Model diagram saved to: optuna_model_architecture.png")

    # Display inline if in Jupyter
    from IPython.display import Image, display

    display(Image("optuna_model_architecture.png"))
except Exception as e:
    print(f"Could not generate model diagram: {e}")
    print("Install graphviz and pydot: pip install pydot graphviz")

## 10. Additional Imports for Training

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts, ExponentialDecay

from mlpng.utils import rmse_metrics
from mlpng.utils.dataloaders import KappaDataset

print("Training imports loaded successfully")

## 11. Prepare Datasets

Create train/validation/test splits using `KappaDataset`.

In [ ]:
# Dataset configuration
DATA_FRACTION = 0.1  # Use 10% of data for faster iteration
BASE_SPLIT = np.array([0.8, 0.1, 0.1], dtype=np.float32)
DEFAULT_DUPLICATES = [10, 2, 2]  # Train, val, test duplicates

# Create dataset
ds = KappaDataset.fromCore(core, x_output="lensed", y_output="fnl")

# Calculate split fractions
fractions = BASE_SPLIT * DATA_FRACTION
cache_dir = os.environ.get("SCRATCH", "/tmp") + "/tf_cache"
os.makedirs(cache_dir, exist_ok=True)

# Include nside in cache filename to avoid shape mismatches
cache_prefix = f"optuna-inspector-n{core.nside}"

# Split into train/val/test
train_ds, val_ds, test_ds = ds.split(
    train_size=float(fractions[0]),
    val_size=float(fractions[1]),
    test_size=float(fractions[2]),
    to_tf=True,
    batch_size=batch_size,
    duplicates=DEFAULT_DUPLICATES,
    cache_dir=cache_dir,
    cache_file=cache_prefix,
    gen_batch_size=max(1, os.cpu_count() or 4),
)

print(f"Data fraction: {DATA_FRACTION * 100:.0f}%")
print(f"Train size: {fractions[0] * 100:.1f}%")
print(f"Val size: {fractions[1] * 100:.1f}%")
print(f"Test size: {fractions[2] * 100:.1f}%")
print(
    f"Duplicates: train={DEFAULT_DUPLICATES[0]}, val={DEFAULT_DUPLICATES[1]}, test={DEFAULT_DUPLICATES[2]}"
)
print(f"Cache prefix: {cache_prefix}")

## 12. Compile and Train the Model

Compile with AdamW optimizer using the best hyperparameters and train.

In [ ]:
# Training configuration
MAX_EPOCHS = 30
PATIENCE = 8

# Estimate steps for learning rate schedule
train_fraction = float(BASE_SPLIT[0] * DATA_FRACTION)
approx_samples = max(1, int(core.total_sims * train_fraction)) * DEFAULT_DUPLICATES[0]
steps_per_epoch = max(1, math.ceil(approx_samples / batch_size))
decay_steps = steps_per_epoch * 2

print(f"Approx training samples: {approx_samples}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Decay steps: {decay_steps}")

# Create learning rate schedule based on best params
if best_params["use_cosine_decay"]:
    lr_schedule = CosineDecayRestarts(
        initial_learning_rate=best_params["initial_lr"],
        first_decay_steps=decay_steps,
        t_mul=2.0,
        m_mul=0.95,
        alpha=0.01,
    )
    print(f"Using CosineDecayRestarts with initial_lr={best_params['initial_lr']:.6f}")
else:
    lr_schedule = ExponentialDecay(
        initial_learning_rate=best_params["initial_lr"],
        decay_steps=decay_steps,
        decay_rate=0.96,
        staircase=True,
    )
    print(f"Using ExponentialDecay with initial_lr={best_params['initial_lr']:.6f}")

# Compile the model
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=best_params["weight_decay"])
model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=rmse_metrics(core.shapes),
)

print(f"\nModel compiled with weight_decay={best_params['weight_decay']:.2e}")

In [ ]:
# Train the model
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nTraining completed!")
print(f"Best val_loss: {min(history.history['val_loss']):.6f}")

## 13. Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RMSE plot for each shape
for shape in core.shapes:
    metric_name = f"rmse_{shape}"
    val_metric_name = f"val_rmse_{shape}"
    if metric_name in history.history:
        axes[1].plot(history.history[metric_name], label=f"Train {shape}")
    if val_metric_name in history.history:
        axes[1].plot(
            history.history[val_metric_name], label=f"Val {shape}", linestyle="--"
        )

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSE")
axes[1].set_title("RMSE per Shape")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Evaluate on Test Set and Plot Predictions

In [ ]:
# Get predictions on test set
preds = model.predict(test_ds, verbose=0)

# Extract ground truth from test dataset
truth = np.concatenate([y.numpy() for _, y in test_ds])

# Flatten if single output
if len(core.shapes) == 1:
    truth = truth.ravel()
    preds = preds.ravel()

# Calculate metrics
error = preds - truth
rmse = np.sqrt(np.mean(error**2))
mae = np.mean(np.abs(error))

print(f"Test Set Metrics:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  Predictions shape: {preds.shape}")
print(f"  Truth shape: {truth.shape}")

In [ ]:
def plot_predictions_analysis(truth, preds, title="", sigma=None):
    """Plot prediction analysis: scatter, residuals histogram, and Q-Q plot."""
    error = preds - truth

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 1. Scatter plot: truth vs predictions
    line = np.array([np.nanmin(truth), np.nanmax(truth)])
    axes[0].scatter(truth, preds, alpha=0.5, s=10)
    axes[0].plot(line, line, "r--", linewidth=2, label="Perfect prediction")
    if sigma is not None:
        axes[0].plot(line, line + sigma, "g--", linewidth=1, label=f"+σ={sigma:.2f}")
        axes[0].plot(line, line - sigma, "g--", linewidth=1, label=f"-σ={sigma:.2f}")
    axes[0].set_xlabel("True fnl")
    axes[0].set_ylabel("Predicted fnl")
    axes[0].set_title(f"{title} Predictions vs Truth")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # 2. Histogram of residuals
    axes[1].hist(error, bins=50, edgecolor="black", alpha=0.7)
    axes[1].axvline(0, color="r", linestyle="--", linewidth=2, label="Zero error")
    if sigma is not None:
        axes[1].axvline(
            sigma, color="g", linestyle="--", linewidth=1, label=f"±σ={sigma:.2f}"
        )
        axes[1].axvline(-sigma, color="g", linestyle="--", linewidth=1)
    axes[1].set_xlabel("Prediction Error (pred - truth)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"{title} Error Distribution")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # 3. Error vs True value
    axes[2].scatter(truth, error, alpha=0.5, s=10)
    axes[2].axhline(0, color="r", linestyle="--", linewidth=2)
    if sigma is not None:
        axes[2].axhline(sigma, color="g", linestyle="--", linewidth=1, label=f"±σ")
        axes[2].axhline(-sigma, color="g", linestyle="--", linewidth=1)
    axes[2].set_xlabel("True fnl")
    axes[2].set_ylabel("Error")
    axes[2].set_title(f"{title} Error vs True Value")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return rmse, mae


# Try to get Fisher sigma from core
try:
    sigma = core.get_likelihoods(lensed=True)[0]
    print(f"Fisher σ (lensed): {sigma:.2f}")
except Exception as e:
    sigma = None
    print(f"Could not get Fisher σ: {e}")

# Plot predictions
plot_predictions_analysis(truth, preds, title="fnl", sigma=sigma)

## 15. phi_scale Inference Loop (No Retraining)

Test the trained model's predictions on data generated with different `phi_scale` values without retraining. This tests the model's robustness to distribution shift in the lensing potential amplitude.

In [ ]:
# phi_scale values to test
PHI_SCALES = [1, 10, 100, 1000]

# Store results for comparison
inference_results = {}

for phi_scale in PHI_SCALES:
    print(f"\n--- phi_scale = {phi_scale} ---")

    # Create dataset with new phi_scale
    ds_phi = KappaDataset.fromCore(
        core, x_output="lensed", y_output="fnl", phi_scale=phi_scale
    )

    # Only create test set for inference (include nside in cache name)
    _, _, test_phi = ds_phi.split(
        train_size=float(fractions[0]),
        val_size=float(fractions[1]),
        test_size=float(fractions[2]),
        to_tf=True,
        batch_size=batch_size,
        duplicates=DEFAULT_DUPLICATES,
        cache_dir=cache_dir,
        cache_file=f"optuna-inspector-n{core.nside}-phi{phi_scale}",
        gen_batch_size=max(1, os.cpu_count() or 4),
    )

    # Get predictions using trained model
    preds_phi = model.predict(test_phi, verbose=1)
    truth_phi = np.concatenate([y.numpy() for _, y in test_phi])

    # Flatten if single output
    if len(core.shapes) == 1:
        truth_phi = truth_phi.ravel()
        preds_phi = preds_phi.ravel()

    # Calculate metrics
    error_phi = preds_phi - truth_phi
    rmse_phi = np.sqrt(np.mean(error_phi**2))
    mae_phi = np.mean(np.abs(error_phi))

    inference_results[phi_scale] = {
        "rmse": rmse_phi,
        "mae": mae_phi,
        "truth": truth_phi,
        "preds": preds_phi,
    }

    print(f"  RMSE: {rmse_phi:.4f}")
    print(f"  MAE:  {mae_phi:.4f}")

print("\n" + "=" * 70)
print("Inference complete!")

In [ ]:
# Plot inference results across phi_scales
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, phi_scale in enumerate(PHI_SCALES):
    result = inference_results[phi_scale]
    truth_phi = result["truth"]
    preds_phi = result["preds"]

    line = np.array([np.nanmin(truth_phi), np.nanmax(truth_phi)])
    axes[idx].scatter(truth_phi, preds_phi, alpha=0.5, s=10)
    axes[idx].plot(line, line, "r--", linewidth=2, label="Perfect")
    axes[idx].set_xlabel("True fnl")
    axes[idx].set_ylabel("Predicted fnl")
    axes[idx].set_title(f'phi_scale = {phi_scale}\nRMSE = {result["rmse"]:.4f}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle(
    f"Model Inference on Different phi_scale (NSide = {core.nside})", fontsize=14
)
plt.tight_layout()
plt.show()

# Summary bar chart
fig, ax = plt.subplots(figsize=(10, 5))
scales = [str(s) for s in PHI_SCALES]
rmses = [inference_results[s]["rmse"] for s in PHI_SCALES]
maes = [inference_results[s]["mae"] for s in PHI_SCALES]

x = np.arange(len(scales))
width = 0.35

bars1 = ax.bar(x - width / 2, rmses, width, label="RMSE")
bars2 = ax.bar(x + width / 2, maes, width, label="MAE")

ax.set_xlabel("phi_scale")
ax.set_ylabel("Error")
ax.set_title(f"Inference Error vs phi_scale (NSide = {core.nside})")
ax.set_xticks(x)
ax.set_xticklabels(scales)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 16. phi_scale Training Loop (Train New Models)

Train fresh models for each `phi_scale` value to see how training performance varies with lensing amplitude.

In [ ]:
# Store training results for each phi_scale
training_results = {}

print("Training new models for each phi_scale value")
print("=" * 70)

for phi_scale in PHI_SCALES:
    print(f"\n{'='*70}")
    print(f"Training model for phi_scale = {phi_scale}")
    print("=" * 70)

    # Create dataset
    ds_phi = KappaDataset.fromCore(
        core, x_output="lensed", y_output="fnl", phi_scale=phi_scale
    )

    train_phi, val_phi, test_phi = ds_phi.split(
        train_size=float(fractions[0]),
        val_size=float(fractions[1]),
        test_size=float(fractions[2]),
        to_tf=True,
        batch_size=batch_size,
        duplicates=DEFAULT_DUPLICATES,
        cache_dir=cache_dir,
        cache_file=f"optuna-inspector-n{core.nside}-train-phi{phi_scale}",
        gen_batch_size=max(1, os.cpu_count() or 4),
    )

    # Build fresh model with best hyperparameters
    model_phi = TunableEncoderFnlModel(
        input_shape=(None, core.npix, core.npols),
        n_outputs=len(core.shapes),
        max_batch_size=batch_size,
        pool_p=best_params["pool_p"],
        transformer_levels=best_params["transformer_levels"],
        num_heads=best_params["num_heads"],
        transformer_layers=best_params["transformer_layers"],
        dropout_rate=best_params["dropout_rate"],
        dense_units=best_params["dense_units"],
        dense_layers=best_params["dense_layers"],
        head_dropout=best_params["head_dropout"],
    ).get_model()

    # Create learning rate schedule
    if best_params["use_cosine_decay"]:
        lr_schedule_phi = CosineDecayRestarts(
            initial_learning_rate=best_params["initial_lr"],
            first_decay_steps=decay_steps,
            t_mul=2.0,
            m_mul=0.95,
            alpha=0.01,
        )
    else:
        lr_schedule_phi = ExponentialDecay(
            initial_learning_rate=best_params["initial_lr"],
            decay_steps=decay_steps,
            decay_rate=0.96,
            staircase=True,
        )

    # Compile
    optimizer_phi = AdamW(
        learning_rate=lr_schedule_phi, weight_decay=best_params["weight_decay"]
    )
    model_phi.compile(
        optimizer=optimizer_phi,
        loss="mse",
        metrics=rmse_metrics(core.shapes),
    )

    # Train
    callbacks_phi = [
        TerminateOnNaN(),
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    history_phi = model_phi.fit(
        train_phi,
        validation_data=val_phi,
        epochs=MAX_EPOCHS,
        steps_per_epoch=steps_per_epoch,
        callbacks=callbacks_phi,
        verbose=1,
    )

    # Evaluate on test set
    preds_phi = model_phi.predict(test_phi, verbose=0)
    truth_phi = np.concatenate([y.numpy() for _, y in test_phi])

    if len(core.shapes) == 1:
        truth_phi = truth_phi.ravel()
        preds_phi = preds_phi.ravel()

    error_phi = preds_phi - truth_phi
    rmse_phi = np.sqrt(np.mean(error_phi**2))
    mae_phi = np.mean(np.abs(error_phi))
    best_val_loss = min(history_phi.history["val_loss"])

    training_results[phi_scale] = {
        "rmse": rmse_phi,
        "mae": mae_phi,
        "best_val_loss": best_val_loss,
        "truth": truth_phi,
        "preds": preds_phi,
        "history": history_phi.history,
    }

    print(f"\n  Results for phi_scale={phi_scale}:")
    print(f"    Best val_loss: {best_val_loss:.6f}")
    print(f"    Test RMSE: {rmse_phi:.4f}")
    print(f"    Test MAE:  {mae_phi:.4f}")

print("\n" + "=" * 70)
print("Training loop complete!")

In [ ]:
# Plot training results across phi_scales - Predictions
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, phi_scale in enumerate(PHI_SCALES):
    result = training_results[phi_scale]
    truth_phi = result["truth"]
    preds_phi = result["preds"]

    line = np.array([np.nanmin(truth_phi), np.nanmax(truth_phi)])
    axes[idx].scatter(truth_phi, preds_phi, alpha=0.5, s=10)
    axes[idx].plot(line, line, "r--", linewidth=2, label="Perfect")
    axes[idx].set_xlabel("True fnl")
    axes[idx].set_ylabel("Predicted fnl")
    axes[idx].set_title(f'phi_scale = {phi_scale}\nRMSE = {result["rmse"]:.4f}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle(f"Model Predictions (NSide = {core.nside})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot training histories for all phi_scales
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, phi_scale in enumerate(PHI_SCALES):
    result = training_results[phi_scale]
    hist = result["history"]

    axes[idx].plot(hist["loss"], label="Train Loss")
    axes[idx].plot(hist["val_loss"], label="Val Loss")
    axes[idx].set_xlabel("Epoch")
    axes[idx].set_ylabel("Loss (MSE)")
    axes[idx].set_title(f"phi_scale = {phi_scale}")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Training Loss Curves per phi_scale", fontsize=14)
plt.tight_layout()
plt.show()

## 17. Compare Results: Inference vs Training

In [ ]:
# Compare inference (no retraining) vs training at each phi_scale
fig, ax = plt.subplots(figsize=(12, 6))

scales = [str(s) for s in PHI_SCALES]
rmse_inference = [inference_results[s]["rmse"] for s in PHI_SCALES]
rmse_trained = [training_results[s]["rmse"] for s in PHI_SCALES]

x = np.arange(len(scales))
width = 0.35

bars1 = ax.bar(
    x - width / 2,
    rmse_inference,
    width,
    label="Inference Only (original model)",
    color="steelblue",
)
bars2 = ax.bar(
    x + width / 2, rmse_trained, width, label="Trained at phi_scale", color="coral"
)

ax.set_xlabel("phi_scale")
ax.set_ylabel("RMSE")
ax.set_title("RMSE Comparison: Inference vs Training at Each phi_scale")
ax.set_xticks(x)
ax.set_xticklabels(scales)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(
        f"{height:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

for bar in bars2:
    height = bar.get_height()
    ax.annotate(
        f"{height:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

# Print summary table
print("\nSummary Table: RMSE by phi_scale")
print("=" * 60)
print(f"{'phi_scale':<12} {'Inference':<15} {'Trained':<15} {'Improvement':<15}")
print("-" * 60)
for phi_scale in PHI_SCALES:
    inf_rmse = inference_results[phi_scale]["rmse"]
    train_rmse = training_results[phi_scale]["rmse"]
    improvement = ((inf_rmse - train_rmse) / inf_rmse) * 100 if inf_rmse > 0 else 0
    print(
        f"{phi_scale:<12} {inf_rmse:<15.4f} {train_rmse:<15.4f} {improvement:<15.2f}%"
    )
print("=" * 60)